Importing Libraries

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight


Importing Data

In [4]:
df = pd.read_csv("../data/processed/modeling_dataset.csv")

Selecting Feature Set

In [5]:

chronic_condition_features = [
    'CVDINFR4',   # Heart attack
    'CVDCRHD4',   # Coronary heart disease
    'CVDSTRK3',   # Stroke
    'ASTHMA3',    # Asthma
    'CHCSCNC1',   # Skin cancer
    'CHCOCNC1',   # Other cancer
    'CHCCOPD3',   # COPD
    'ADDEPEV3',   # Depression
    'CHCKDNY2'    # Kidney disease
]

df['chronic_count'] = df[chronic_condition_features].sum(axis=1)

features = [
    '_STATE',
    'SEXVAR',
    'MARITAL',
    'EDUCA',
    'RENTHOM1',
    '_AGE80',
    '_RACEGR3',
    'INCOME3',
    'EMPLOY1',
    'VETERAN3',
    'PRIMINS2',
    'PERSDOC3',
    'CHECKUP1',
    'GENHLTH',
    'PHYSHLTH',
    'MENTHLTH',
    'POORHLTH',
    'CVDINFR4',
    'CVDCRHD4',
    'CVDSTRK3',
    'ASTHMA3',
    'CHCSCNC1',
    'CHCOCNC1',
    'CHCCOPD3',
    'ADDEPEV3',
    'CHCKDNY2',
    'EXERANY2',
    'SMOKE100',
    '_BMI5',
    'LASTDEN4',
    'RMVTETH4',
    'chronic_count'
]


X = df[features]
y = df['MEDCOST1_binary']

Splitting dataset and creating weights

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print(scale_pos_weight)


classes = np.unique(y_train)

weights = compute_class_weight(class_weight="balanced",classes=classes,y=y_train)

class_weights = dict(zip(classes, weights))


print(class_weights)

9.515912366676275
{np.int64(0): np.float64(0.5525435692063483), np.int64(1): np.float64(5.257956183338138)}


Creating Model Pipelines

In [ ]:
log_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier",
         LogisticRegression(
             class_weight=class_weights,
             max_iter=2000,
             random_state=42
         ))
    ]
)

rf_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
             n_estimators=400,
             class_weight=class_weights,
             random_state=42
         ))
    ]
)

xgb_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", XGBClassifier(
             n_estimators=400,
             max_depth=6,
             learning_rate=0.05,
             subsample=0.8,
             colsample_bytree=0.8,
             scale_pos_weight=scale_pos_weight,
             random_state=42,
             eval_metric="logloss"
         ))
    ]
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Logisitc Regression

In [ ]:
log_model.fit(X_train, y_train)
y_prob = log_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.3).astype(int)

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
log_results = cross_validate(log_model, X, y, cv=cv, scoring=scoring)

print("Logistic Regression Cross Validation")
for metric in scoring:
    print(metric, round(log_results[f"test_{metric}"].mean(), 3))

Logistic Regression
Accuracy: 0.5862280701754385
Precision: 0.17531002122667858
Recall: 0.9046466044044736
F1: 0.29370367597514413
ROC-AUC: 0.835747996821674
Logistic Regression
accuracy 0.77
precision 0.255
recall 0.737
f1 0.378
roc_auc 0.836


Random Forest

In [ ]:
rf_model.fit(X_train, y_train)
y_prob = rf_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.3).astype(int)

print("Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

rf_results = cross_validate(rf_model, X, y, cv=cv, scoring=scoring)

print("Random Forest Cross Validation")
for metric in scoring:
    print(metric, round(rf_results[f"test_{metric}"].mean(), 3))

Random Forest
Accuracy: 0.9058552631578948
Precision: 0.5076463350325189
Recall: 0.33298743226104
F1: 0.40217239938727195
ROC-AUC: 0.8470885463811486
Random Forest
accuracy 0.912
precision 0.705
recall 0.133
f1 0.224
roc_auc 0.848


XGBoost

In [ ]:
xgb_model.fit(X_train, y_train)
y_prob = xgb_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.7).astype(int)

print("XGBoost")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

xgb_results = cross_validate(xgb_model, X, y, cv=cv, scoring=scoring)

print("XGBoost Cross Validation")
for metric in scoring:
    print(metric, round(xgb_results[f"test_{metric}"].mean(), 3))

XGBoost
Accuracy: 0.8726754385964912
Precision: 0.38212882008502447
Recall: 0.5492909028017987
F1: 0.4507095553453169
ROC-AUC: 0.8570563438031861
XGBoost
accuracy 0.776
precision 0.266
recall 0.774
f1 0.396
roc_auc 0.857


Retrieving Feature Importance

In [ ]:
xgb = xgb_model.named_steps["classifier"]

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb.feature_importances_
})

importance = importance.sort_values("Importance", ascending=False)
print(importance.head(15))

     Feature  Importance
10  PRIMINS2    0.120621
15  MENTHLTH    0.083884
7    INCOME3    0.078486
16  POORHLTH    0.072036
4   RENTHOM1    0.068975
5     _AGE80    0.064433
29  LASTDEN4    0.060534
13   GENHLTH    0.054619
12  CHECKUP1    0.044191
24  ADDEPEV3    0.032226
6   _RACEGR3    0.031204
11  PERSDOC3    0.028642
8    EMPLOY1    0.025736
14  PHYSHLTH    0.021618
30  RMVTETH4    0.020222


Quick Comparison of Cross Validation results

In [20]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        log_results["test_accuracy"].mean(),
        rf_results["test_accuracy"].mean(),
        xgb_results["test_accuracy"].mean()
    ],
    "Precision": [
        log_results["test_precision"].mean(),
        rf_results["test_precision"].mean(),
        xgb_results["test_precision"].mean()
    ],
    "Recall": [
        log_results["test_recall"].mean(),
        rf_results["test_recall"].mean(),
        xgb_results["test_recall"].mean()
    ],
    "F1": [
        log_results["test_f1"].mean(),
        rf_results["test_f1"].mean(),
        xgb_results["test_f1"].mean()
    ],
    "ROC-AUC": [
        log_results["test_roc_auc"].mean(),
        rf_results["test_roc_auc"].mean(),
        xgb_results["test_roc_auc"].mean()
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.769768,0.254597,0.737149,0.378474,0.836093
1,Random Forest,0.912289,0.705040,0.133478,0.224448,0.847691
2,XGBoost,0.775567,0.266105,0.773701,0.396007,0.857410


The XGBoost Model seems to perform the best according to the ROC-AUC score and F1 score being the highest. The logistic regression model had the highest recall when given a lower threshold while the Random Forest model was the weakest overall. The most important features used to determine at-risk people was their insurance status, mental health, and income. In other words, it generally seems true that the most important factors in deciding whether someone avoided medical care due to cost is their financial status and their current health.